# 스팸 분류 - 데이터 탐색(EDA) & 전처리 시각화

이 프로젝트는 **언어별로 별도 모델**을 쓴다: 한국어=글자 단위 LSTM(F1 0.968), 영어=단어 단위 LSTM(F1 0.957).
글자 단위는 영어에서 3회(Enron/spam_or_not_spam/completeSpamAssassin) 실패해 단어 단위로 분리했다
(자세한 경위는 `docs/CHANGELOG.md` 2026-07-10 참고). 이 노트북은 **두 언어의 전처리·데이터를 각각** 눈으로 확인한다.

1. 데이터 로드 & 개요 (한국어 `spam.csv` / 영어 `spam_en.csv`)
2. 클래스/언어 분포
3. 텍스트 길이 분포 (한국어=글자수 MAX_LEN 250 / 영어=단어수 MAX_LEN 300 근거)
4. 전처리 단계별 시연 (한국어: 글자 단위 / 영어: 단어 단위)
5. 어휘사전 구축 (언어별)
6. 스팸 vs 정상 특징 토큰 비교 (언어별)
7. 인코딩된 시퀀스 예시 (언어별)
8. 학습된 모델 예측 & 혼동행렬 (한국어 데모셋 + 영어 예시 문장, 언어 라우터)

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# 프로젝트 루트 탐색 (data 폴더가 있는 상위 디렉터리)
ROOT = Path.cwd()
while not (ROOT / 'data').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'
print('프로젝트 루트:', ROOT)

# 한글 폰트 (Windows: Malgun Gothic)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (8, 4)

## 1. 데이터 로드 & 개요

한국어(`data/spam.csv`)와 영어(`data/spam_en.csv`)는 **완전히 분리된 학습셋**이다
(언어별로 별도 모델·별도 전처리를 쓰기 때문). EDA도 각각 본다.

In [ ]:
df_ko = pd.read_csv(DATA / 'spam.csv')
df_en = pd.read_csv(DATA / 'spam_en.csv')

print('=== 한국어 (spam.csv) ===')
print('총 건수:', len(df_ko), '| 컬럼:', list(df_ko.columns))
display(df_ko.head(3))
print(df_ko['label'].value_counts())

print('\n=== 영어 (spam_en.csv) ===')
print('총 건수:', len(df_en), '| 컬럼:', list(df_en.columns))
display(df_en.head(3))
print(df_en['label'].value_counts())

## 2. 클래스 / 언어 분포 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df_ko['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#55A868', '#C44E52'])
axes[0].set_title('한국어 클래스 분포 (ham / spam)')
axes[0].set_ylabel('건수')

df_en['label'].value_counts().plot(kind='bar', ax=axes[1], color=['#8172B3', '#C44E52'])
axes[1].set_title('영어 클래스 분포 (ham / spam)')

combo = pd.DataFrame({
    '한국어': df_ko['label'].value_counts(),
    '영어': df_en['label'].value_counts(),
})
combo.plot(kind='bar', ax=axes[2], color=['#55A868', '#8172B3'])
axes[2].set_title('언어별 건수 비교')

for ax in axes:
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 3. 텍스트 길이 분포 (→ MAX_LEN 근거)

**한국어는 글자 단위**라 글자 수, **영어는 단어 단위**라 단어 수 분포를 각각 본다.

In [ ]:
from src.preprocessing import clean_text as clean_ko
from src.preprocessing_en import clean_text as clean_en

df_ko['len'] = df_ko['text'].str.len()
df_en['wlen'] = df_en['text'].apply(lambda t: len(clean_en(str(t))))

print('한국어 글자수:', df_ko['len'].describe()[['mean', '50%', 'max']].to_dict())
print('영어 단어수  :', df_en['wlen'].describe()[['mean', '50%', 'max']].to_dict())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df_ko['len'].clip(upper=600), bins=50, color='#55A868')
axes[0].axvline(250, color='red', linestyle='--', label='MAX_LEN=250')
axes[0].set_title('한국어: 글자 수 분포'); axes[0].set_xlabel('글자 수(600 클립)'); axes[0].legend()

axes[1].hist(df_en['wlen'].clip(upper=800), bins=50, color='#8172B3')
axes[1].axvline(300, color='red', linestyle='--', label='MAX_LEN=300')
axes[1].set_title('영어: 단어 수 분포'); axes[1].set_xlabel('단어 수(800 클립)'); axes[1].legend()
plt.tight_layout(); plt.show()

## 4. 전처리 과정 단계별 시연

- 한국어 `clean_text()` (글자 단위): 소문자화 → URL→`<url>` → 숫자→`<num>` → **글자 단위** 분해
- 영어 `clean_text()` (단어 단위): 동일한 URL/숫자 치환 → **단어(알파벳 덩어리) 단위** 분해

글자 단위는 영어에서 실패했었고(일반 이메일 오판), 단어 단위로 바꿔 해결했다.

In [ ]:
print('--- 한국어 (글자 단위) ---')
for s in ['(광고) 저금리 대출 1억 지금 신청 http://bit.ly/xz39',
          '내일 오후 2시 회의 참석 부탁드립니다']:
    toks = clean_ko(s)
    print('원문 :', s)
    print('토큰 :', toks[:20], '...' if len(toks) > 20 else '')
    print('토큰수:', len(toks), '\n')

print('--- 영어 (단어 단위) ---')
for s in ['Subject: free money click now !!! http://bit.ly/xz39',
          "Thank you for your order, it won't arrive until Monday."]:
    toks = clean_en(s)
    print('원문 :', s)
    print('토큰 :', toks)
    print('토큰수:', len(toks), '\n')

## 5. 어휘사전(vocab) 구축 (언어별)

In [ ]:
from src.preprocessing import build_vocab as build_vocab_ko
from src.preprocessing_en import build_vocab as build_vocab_en

vocab_ko = build_vocab_ko(df_ko['text'].tolist())
vocab_en = build_vocab_en(df_en['text'].tolist())
print('한국어 어휘사전 크기(글자):', len(vocab_ko))
print('영어   어휘사전 크기(단어):', len(vocab_en))

cnt_ko = Counter(); [cnt_ko.update(clean_ko(t)) for t in df_ko['text']]
cnt_en = Counter(); [cnt_en.update(clean_en(t)) for t in df_en['text']]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, cnt, title, color in [(axes[0], cnt_ko, '한국어 최빈 글자 20개', '#55A868'),
                              (axes[1], cnt_en, '영어 최빈 단어 20개', '#8172B3')]:
    top = cnt.most_common(20)
    labels, values = zip(*top)
    ax.bar(range(len(labels)), values, color=color)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_title(title)
plt.tight_layout(); plt.show()

## 6. 스팸 vs 정상 - 특징적인 토큰 비교 (언어별)

각 클래스에서 상대적으로 더 자주 등장하는 토큰을 본다 (스팸성 신호 확인).

In [ ]:
def token_counter(texts, clean_fn):
    c = Counter()
    for t in texts:
        c.update(set(clean_fn(t)))   # 문서 단위 등장(중복 제거)
    return c

def top_signal_tokens(df, clean_fn, min_count=20, topn=15):
    spam_c = token_counter(df[df.label == 'spam']['text'], clean_fn)
    ham_c = token_counter(df[df.label == 'ham']['text'], clean_fn)
    n_spam = (df.label == 'spam').sum(); n_ham = (df.label == 'ham').sum()
    rows = []
    for tok in set(spam_c) | set(ham_c):
        s, h = spam_c.get(tok, 0), ham_c.get(tok, 0)
        if s + h < min_count:
            continue
        ratio = (s / n_spam) / ((s / n_spam) + (h / n_ham) + 1e-9)
        rows.append((tok, ratio))
    top_spam = sorted(rows, key=lambda x: -x[1])[:topn]
    top_ham = sorted(rows, key=lambda x: x[1])[:topn]
    return [t for t, _ in top_spam], [t for t, _ in top_ham]

ko_spam_tok, ko_ham_tok = top_signal_tokens(df_ko, clean_ko)
en_spam_tok, en_ham_tok = top_signal_tokens(df_en, clean_en, min_count=10)

print('[한국어] 스팸 신호 글자 TOP15:', ko_spam_tok)
print('[한국어] 정상 신호 글자 TOP15:', ko_ham_tok)
print('\n[영어] 스팸 신호 단어 TOP15  :', en_spam_tok)
print('[영어] 정상 신호 단어 TOP15  :', en_ham_tok)

## 7. 인코딩된 시퀀스 예시 (언어별)

토큰을 정수 인덱스로 바꾸고 고정 길이로 패딩한 결과 (LSTM 입력 형태).

In [ ]:
from src.preprocessing import encode as encode_ko
from src.preprocessing_en import encode as encode_en

ex_ko = '(광고) 무료 상품 당첨 지금 클릭'
ids_ko = encode_ko(ex_ko, vocab_ko, max_len=20)
print('[한국어] 원문:', ex_ko)
print('[한국어] 인덱스(앞 20):', ids_ko)

ex_en = 'Congratulations you won a free prize click now'
ids_en = encode_en(ex_en, vocab_en, max_len=20)
print('\n[영어] 원문:', ex_en)
print('[영어] 인덱스(앞 20):', ids_en)
print('\n0 = <pad> (뒤쪽 채움), 1 = <unk>')

## 8. 학습된 모델 예측 & 혼동행렬

`python src/train.py` / `python src/train_en.py` 로 두 모델을 각각 학습해야 실행된다.
한국어는 저장된 데모셋(`demo_samples.csv`, 학습 미사용), 영어는 검증에 썼던 예시 문장으로 시연한다.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print('=== 한국어 데모셋 ===')
try:
    from src.predict import SpamPredictor
    predictor_ko = SpamPredictor()
    demo = pd.read_csv(DATA / 'synthetic' / 'demo_samples.csv')
    demo['pred'] = demo['text'].apply(lambda t: predictor_ko.predict(t)[0])
    acc = (demo['pred'] == demo['label']).mean()
    print(f'데모 샘플 정확도: {acc:.3f}  (n={len(demo)})')
    cm = confusion_matrix(demo['label'], demo['pred'], labels=['ham', 'spam'])
    ConfusionMatrixDisplay(cm, display_labels=['ham', 'spam']).plot(cmap='Greens')
    plt.title('한국어 데모 샘플 혼동행렬'); plt.show()
except SystemExit as e:
    print('한국어 모델이 없습니다. 먼저 python src/train.py 실행:', e)

In [ ]:
print('=== 영어 예시 문장 (학습에 쓰이지 않은 held-out 문장) ===')
en_examples = [
    ('Hey, are we still meeting for lunch tomorrow?', 'ham'),
    ('Thank you for your order. Your package will arrive on Monday.', 'ham'),
    ('Please find attached the quarterly report for your review.', 'ham'),
    ('Happy birthday! Hope you have a wonderful day.', 'ham'),
    ('Your appointment is confirmed for 3pm on Friday.', 'ham'),
    ('Congratulations! You won $1000000. Click here to claim now!', 'spam'),
    ('Cheap meds online, no prescription needed, order now!', 'spam'),
    ('You are pre-approved for a $50000 loan, apply now!', 'spam'),
]
try:
    from src.predict_en import EnglishSpamPredictor
    predictor_en = EnglishSpamPredictor()
    edf = pd.DataFrame(en_examples, columns=['text', 'label'])
    edf['pred'] = edf['text'].apply(lambda t: predictor_en.predict(t)[0])
    acc = (edf['pred'] == edf['label']).mean()
    print(f'예시 문장 정확도: {acc:.3f}  (n={len(edf)})')
    cm = confusion_matrix(edf['label'], edf['pred'], labels=['ham', 'spam'])
    ConfusionMatrixDisplay(cm, display_labels=['ham', 'spam']).plot(cmap='Purples')
    plt.title('영어 예시 문장 혼동행렬'); plt.show()
except SystemExit as e:
    print('영어 모델이 없습니다. 먼저 python src/train_en.py 실행:', e)

In [ ]:
# 언어 자동 라우터로 직접 테스트해보기 (한/영 섞어서 넣어도 알아서 판정)
try:
    from src.predict_router import predict_email_tier_auto
    for t in ['안녕하세요 회의 자료 첨부합니다', '무료 코인 10배 수익 리딩방 입장',
              'Thank you for your help with the report', 'Free prize! Click now to claim!!']:
        tier, prob, lang = predict_email_tier_auto(body=t)
        print(f'[{lang}] {tier:6} 스팸확률 {prob:.1%}  | {t}')
except SystemExit as e:
    print('모델이 없습니다:', e)